# Text Generation

In this section we go only through n-gram model. Other methods are presented as demo of working open source examples. Learning takes too long.

## N-gram model

In this example, we use the Wall Street Journal corpus to generate new sentences. The corpus is available as a part of NLTK library. This example is based on [Erroll Wood's work](https://github.com/errollw/gengram).

In [1]:
from nltk.book import *

wall_street = text7.tokens

*** Introductory Examples for the NLTK Book ***
Loading text1, ..., text9 and sent1, ..., sent9
Type the name of the text or sentence to view it.
Type: 'texts()' or 'sents()' to list the materials.
text1: Moby Dick by Herman Melville 1851
text2: Sense and Sensibility by Jane Austen 1811
text3: The Book of Genesis
text4: Inaugural Address Corpus
text5: Chat Corpus
text6: Monty Python and the Holy Grail
text7: Wall Street Journal
text8: Personals Corpus
text9: The Man Who Was Thursday by G . K . Chesterton 1908


We need to clean the text up and delete all meaningless words/characters. The easiest way is to use regular expressions:

In [2]:
import re

tokens = wall_street

def cleanup():
    compiled_pattern = re.compile("^[a-zA-Z0-9.!?]")
    clean = list(filter(compiled_pattern.match,tokens))
    return clean
tokens = cleanup()

In [3]:
print(len(tokens))
tokens

85828


['Pierre',
 'Vinken',
 '61',
 'years',
 'old',
 'will',
 'join',
 'the',
 'board',
 'as',
 'a',
 'nonexecutive',
 'director',
 'Nov.',
 '29',
 '.',
 'Mr.',
 'Vinken',
 'is',
 'chairman',
 'of',
 'Elsevier',
 'N.V.',
 'the',
 'Dutch',
 'publishing',
 'group',
 '.',
 'Rudolph',
 'Agnew',
 '55',
 'years',
 'old',
 'and',
 'former',
 'chairman',
 'of',
 'Consolidated',
 'Gold',
 'Fields',
 'PLC',
 'was',
 'named',
 'a',
 'nonexecutive',
 'director',
 'of',
 'this',
 'British',
 'industrial',
 'conglomerate',
 '.',
 'A',
 'form',
 'of',
 'asbestos',
 'once',
 'used',
 'to',
 'make',
 'Kent',
 'cigarette',
 'filters',
 'has',
 'caused',
 'a',
 'high',
 'percentage',
 'of',
 'cancer',
 'deaths',
 'among',
 'a',
 'group',
 'of',
 'workers',
 'exposed',
 'to',
 'it',
 'more',
 'than',
 '30',
 'years',
 'ago',
 'researchers',
 'reported',
 '0',
 '.',
 'The',
 'asbestos',
 'fiber',
 'crocidolite',
 'is',
 'unusually',
 'resilient',
 'once',
 'it',
 'enters',
 'the',
 'lungs',
 'with',
 'even',
 '

The next step is to build ngrams. It means that we group the tokens into a list of three that are placed next to each other. You can print the ngrams.

In [4]:
def build_ngrams():
    ngrams = []
    for i in range(len(tokens)-N+1):
        ngrams.append(tokens[i:i+N])
    return ngrams

In [5]:
N=3
build_ngrams()

[['Pierre', 'Vinken', '61'],
 ['Vinken', '61', 'years'],
 ['61', 'years', 'old'],
 ['years', 'old', 'will'],
 ['old', 'will', 'join'],
 ['will', 'join', 'the'],
 ['join', 'the', 'board'],
 ['the', 'board', 'as'],
 ['board', 'as', 'a'],
 ['as', 'a', 'nonexecutive'],
 ['a', 'nonexecutive', 'director'],
 ['nonexecutive', 'director', 'Nov.'],
 ['director', 'Nov.', '29'],
 ['Nov.', '29', '.'],
 ['29', '.', 'Mr.'],
 ['.', 'Mr.', 'Vinken'],
 ['Mr.', 'Vinken', 'is'],
 ['Vinken', 'is', 'chairman'],
 ['is', 'chairman', 'of'],
 ['chairman', 'of', 'Elsevier'],
 ['of', 'Elsevier', 'N.V.'],
 ['Elsevier', 'N.V.', 'the'],
 ['N.V.', 'the', 'Dutch'],
 ['the', 'Dutch', 'publishing'],
 ['Dutch', 'publishing', 'group'],
 ['publishing', 'group', '.'],
 ['group', '.', 'Rudolph'],
 ['.', 'Rudolph', 'Agnew'],
 ['Rudolph', 'Agnew', '55'],
 ['Agnew', '55', 'years'],
 ['55', 'years', 'old'],
 ['years', 'old', 'and'],
 ['old', 'and', 'former'],
 ['and', 'former', 'chairman'],
 ['former', 'chairman', 'of'],
 ['chai

The next step is to calculate the frequency of tokens in each ngram and sum if there are more than one tokens related to a ngram. There are 85826 ngrams and 54677 frequency ngrams.

In [9]:
def ngram_freqs(ngrams):
    counts = {}

    for ngram in ngrams:
        token_seq  = SEP.join(ngram[:-1])
        last_token = ngram[-1]

        if token_seq not in counts:
            counts[token_seq] = {}

        if last_token not in counts[token_seq]:
            counts[token_seq][last_token] = 0

        counts[token_seq][last_token] += 1;

    return counts

Example:

In [10]:
ngrams = [
    ("I", "love", "cats"),
    ("I", "love", "dogs"),
    ("I", "love", "cats")
]
ngram_freqs(ngrams)

NameError: name 'SEP' is not defined

We choose the next word by using the most recent tokens and adds it.

In [12]:
def next_word(text, N, counts):

    token_seq = SEP.join(text.split()[-(N-1):]);
    choices = counts[token_seq].items();

    total = sum(weight for choice, weight in choices)
    r = random.uniform(0, total)
    upto = 0
    for choice, weight in choices:
        upto += weight;
        if upto > r: return choice
    assert False # should not reach here

We need to setup a few parameters like the windows size N, the number of sentences that we want to generate and start of the sentence that we want to generate. The sentence start string are N-1 words that exists in our ngrams list.

In [13]:
import random

N=3

SEP=" "

sentence_count=5

ngrams = build_ngrams()
start_seq="We have"

counts = ngram_freqs(ngrams)

if start_seq is None: start_seq = random.choice(list(counts.keys()))
generated = start_seq.lower();

sentences = 0
while sentences < sentence_count:
    generated += SEP + next_word(generated, N, counts)
    sentences += 1 if generated.endswith(('.','!', '?')) else 0

print(generated)

we have to scrape says Mr. Hammond a retired water-authority worker . We would like to get a disproportionate share of a specialist and once one himself Mr. Phelan what would be an 11 increase over fiscal 1989 . The competition has cultivated a much more than 3 billion from 2.8 billion and Minnesota were among the highest savings .
